# 10. pandas: Reading and Writing Data

In the previous chapter, you became familiar with the pandas library and with the basic functionalities that it provides for data analysis. You saw that dataframe and series are the heart of this library. These are the material on which to perform all data manipulations, calculations, and analysis.

In this chapter, you will see all of the tools provided by pandas for reading data stored in many types of media (such as files and databases). In parallel, you will also see how to write data structures directly on these formats, without worrying too much about the technologies used.

This chapter focuses on a series of I/O API functions that pandas provides to read and write data directly as dataframe objects. We start by looking at text files.

## 10.1. I/O API Tools

pandas is a library specialized for data analysis, so you expect that it is mainly focused on calculation and data processing. The processes of writing and reading data from/to external files can be considered part of data processing. In fact, you will see how, even at this stage, you can perform some operations in order to prepare the incoming data for manipulation.

Thus, this step is very important for data analysis and therefore a specific tool for this purpose must be present in the library pandas—a set of functions called I/O API. These functions are divided into two main categories: readers and writers.

<img src="i-o_api.png" width="" align="" />

## 10.2. CSV and Textual Files

Everyone has become accustomed over the years to writing and reading files in text form. In particular, data are generally reported in tabular form. If the values in a row are separated by commas, you have the CSV (comma-separated values) format, which is perhaps the best-known and most popular format.

Other forms of tabular data can be separated by spaces or tabs and are typically contained in text files of various types (generally with the .txt extension).

This type of file is the most common source of data and is easier to transcribe and interpret. In this regard, pandas provides a set of functions specific for this type of file.

- read_csv

- read_table

- to_csv

## 10.3. Reading Data in CSV or Text Files

From experience, the most common operation of a person approaching data analysis is to read the data contained in a CSV file, or at least in a text file.

But before you start dealing with files, you need to import the following libraries.

In [ ]:
import numpy as np
import pandas as pd

In order to see how pandas handles this kind of data, we’ll start with a small CSV file in the working directory called ch10_01.csv.

Since this file is comma-delimited, you can use the read_csv() function to read its content and convert it to a dataframe object.

In [ ]:
csvframe = pd.read_csv('./ch10_01.csv')
csvframe

As you can see, reading the data in a CSV file is rather trivial. CSV files are tabulated data in which the values on the same column are separated by commas. Since CSV files are considered text files, you can also use the read_table() function, but specify the delimiter.

In [ ]:
pd.read_table('./ch10_01.csv',sep=',')

In this example, you can see that in the CSV file, headers that identify all the columns are in the first row. But this is not a general case; it often happens that the tabulated data begin directly in the first line.

In [ ]:
pd.read_csv('./ch10_02.csv')

In this case, you could make sure that it is pandas that assigns the default names to the columns by setting the header option to None.

In [ ]:
pd.read_csv('./ch10_02.csv', header=None)

In addition, you can specify the names directly by assigning a list of labels to the names option.

In [ ]:
pd.read_csv('./ch10_02.csv', names=['white','red','blue','green','animal'])

In more complex cases, in which you want to create a dataframe with a hierarchical structure by reading a CSV file, you can extend the functionality of the read_csv() function by adding the index_col option, assigning all the columns to be converted into indexes.

In [ ]:
pd.read_csv('./ch10_03.csv', index_col=['color','status'])

### 10.3.1. Using RegExp to Parse TXT Files

In other cases, it is possible that the files on which to parse the data do not show separators well defined as a comma or a semicolon. In these cases, the regular expressions come to our aid. In fact, you can specify a regexp within the read_table() function using the sep option.

To better understand regexp and understand how you can apply it as criteria for value separation, let’s start with a simple case. For example, suppose that your TXT file has values that are separated by spaces or tabs in an unpredictable order. In this case, you have to use the regexp, because that’s the only way to take into account both separator types. You can do that using the wildcard /s*. /s stands for the space or tab
character (if you want to indicate a tab, you use /t), while the asterisk indicates that there may be multiple characters. That is, the values may be separated by more spaces or more tabs.

<img src="metacharacters.png" width="" align="" />

Take for example an extreme case in which we have the values separated by tabs or spaces in a random order.

In [ ]:
pd.read_table('./ch10_04.txt',sep='\s+', engine='python')

As you can see, the result is a perfect dataframe in which the values are perfectly ordered.

Now you will see an example that may seem strange or unusual, but it is not as rare as it may seem. This example can be very helpful in understanding the high potential of a regexp. In fact, you might typically think of separators as special characters like commas, spaces, tabs, etc., but in reality you can consider separator characters like alphanumeric characters, or for example, integers such as 0.

In this example, you need to extract the numeric part from a TXT file, in which there is a sequence of characters with numerical values and the literal characters are completely fused.

Remember to set the header option to None whenever the column headings are not present in the TXT file.

In [ ]:
pd.read_table('./ch10_05.txt', sep='\D+', header=None, engine='python')

Another fairly common event is to exclude lines from parsing. In fact you do not always want to include headers or unnecessary comments contained in a file. With the skiprows option, you can exclude all the lines you want, just assigning an array containing the line numbers to not consider in parsing.

Pay attention when you are using this option. If you want to exclude the first five lines, you have to write skiprows = 5, but if you want to rule out the fifth line, you have to write skiprows = [5].

In [ ]:
pd.read_table('./ch10_06.txt',sep=',',skiprows=[0,1,3,6])

### 10.3.2. Reading TXT Files Into Parts

When large files are processed, or when you are only interested in portions of these files, you often need to read the file into portions (chunks). This is both to apply any iterations and because we are not interested in parsing the entire file.

If, for example, you wanted to read only a portion of the file, you can explicitly specify the number of lines on which to parse. Thanks to the nrows and skiprows options, you can select the starting line n (n = SkipRows) and the lines to be read after it (nrows = i).

In [ ]:
pd.read_csv('ch10_02.csv',skiprows=[2],nrows=3,header=None)

Another interesting and fairly common operation is to split into portions that part of the text on which you want to parse. Then, for each portion a specific operation may be carried out, in order to obtain an iteration, portion by portion.

For example, you want to add the values in a column every three rows and then insert these sums in a series. This example is trivial and impractical but is very simple to understand, so once you have learned the underlying mechanism, you will be able to apply it in more complex cases.

In [ ]:
out = pd.Series()
i = 0
pieces = pd.read_csv('./ch10_01.csv',chunksize=3)
for piece in pieces:
    out.at[i] = piece['white'].sum()
    i = i + 1
out

### 10.3.3. Writing Data in CSV

In addition to reading the data contained in a file, it’s also common to write a data file produced by a calculation, or in general the data contained in a data structure.

For example, you might want to write the data contained in a dataframe to a CSV file. To do this writing process, you use the to_csv() function, which accepts as an argument the name of the file you generate.

In [ ]:
frame = pd.DataFrame(np.arange(16).reshape((4,4)),
    index = ['red', 'blue', 'yellow', 'white'],
    columns = ['ball', 'pen', 'pencil', 'paper'])
frame.to_csv('./ch10_07.csv')

If you open the new file called ch10_07.csv generated by the pandas library, you will see data

As you can see from the previous example, when you write a dataframe to a file, indexes and columns are marked on the file by default. This default behavior can be changed by setting the two options index and header to False.

In [ ]:
frame.to_csv('./ch10_07b.csv', index=False, header=False)

One point to remember when writing files is that NaN values present in a data structure are shown as empty fields in the file.

In [ ]:
frame2 = pd.DataFrame([[6,np.nan,np.nan,6,np.nan],
    [np.nan,np.nan,np.nan,np.nan,np.nan],
    [np.nan,np.nan,np.nan,np.nan,np.nan],
    [20,np.nan,np.nan,20.0,np.nan],
    [19,np.nan,np.nan,19.0,np.nan]
    ],
    index=['blue','green','red','white','yellow'],
    columns=['ball','mug','paper','pen','pencil'])
frame2
frame2.to_csv('./ch10_08.csv')

However, you can replace this empty field with a value to your liking using the na_rep option in the to_csv() function. Common values include NULL, 0, or the same NaN.

In [ ]:
frame2.to_csv('./ch10_09.csv', na_rep ='NaN')